### Install Dependencies & Authenticate

In [ ]:
!pip install -q transformers datasets accelerate torch evaluate langfuse huggingface_hub

from google.colab import userdata
from huggingface_hub import login
import torch
import os

# Load Hugging Face token securely from Colab Secrets
hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

# Check GPU availability
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.get_device_name(0)}")

#### Configure Langfuse Environment Variables

In [ ]:
from google.colab import userdata
from langfuse import Langfuse
import os

# Load Langfuse credentials securely from Colab Secrets
os.environ["LANGFUSE_PUBLIC_KEY"] = userdata.get('LANGFUSE_PUBLIC_KEY')
os.environ["LANGFUSE_SECRET_KEY"] = userdata.get('LANGFUSE_SECRET_KEY')
os.environ["LANGFUSE_HOST"] = "https://cloud.langfuse.com"

# Initialize Langfuse client
langfuse = Langfuse()
print("Langfuse telemetry initialized securely.")

#### Load Dataset & Tokenizer

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

MODEL_ID = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=userdata.get('HF_TOKEN'))

# Load parallel dataset
dataset = load_dataset("wmt14", "es-en", split="train[:500]")

def preprocess_multilingual(examples):
    inputs = [ex["en"] for ex in examples["translation"]]
    targets = [ex["es"] for ex in examples["translation"]]
    
    tokenizer.src_lang = "eng_Latn"
    model_inputs = tokenizer(inputs, max_length=128, truncation=True)
    labels = tokenizer(text_target=targets, max_length=128, truncation=True)
    
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = dataset.map(preprocess_multilingual, batched=True)
print("Dataset tokenized and ready for training.")

#### Initialize Model and Run Fine-Tuning with Seq2SeqTrainer

In [ ]:
from transformers import AutoModelForSeq2SeqLM, Seq2SeqTrainer, Seq2SeqTrainingArguments

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID, token=userdata.get('HF_TOKEN'))

training_args = Seq2SeqTrainingArguments(
    output_dir="./models/nllb-multilingual-support",
    evaluation_strategy="no",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=1,
    fp16=True,
    push_to_hub=True,
    hub_model_id="jameskingsley/nllb-multilingual-support-finetuned"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets,
    tokenizer=tokenizer,
)

print("Starting model fine-tuning...")
trainer.train()

# Push final model weights to Hugging Face Hub
trainer.push_to_hub()
print("Model fine-tuned and successfully pushed to Hugging Face!")

#### Log Training Session Metadata to Langfuse

In [ ]:
trace = langfuse.trace(
    name="multilingual_fine_tuning_run",
    metadata={
        "base_model": MODEL_ID,
        "target_languages": ["eng_Latn", "spa_Latn", "fra_Latn", "deu_Latn", "pcm_Latn"],
        "target_repo": "jameskingsley/nllb-multilingual-support-finetuned"
    }
)

trace.generation(
    name="finetune_job",
    model=MODEL_ID,
    output="Multilingual customer support model trained and pushed successfully."
)

langfuse.flush()
print("Training session telemetry logged to Langfuse.")